# KWIC: Reading What a Count Points At

**Computational Text Analysis Pack · Module 05 — Concordance and Collocation**

*Technique:* Keyword-in-context; using a count to direct close reading

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/05a_kwic_concordance.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The concordance is the oldest tool in this discipline — monks built them for the
Bible by hand, one line per occurrence, keyword down the middle.

It is also the answer to the objection that counting strips away meaning. A count
tells you *where* a word matters; KWIC puts the context back. Used together they
are close reading with a search strategy, which is the honest description of most
good digital humanities work.

## The tooling

Standard library again — concordancing and collocation are slicing and counting.
**AntConc** and **CQPweb** are the dedicated tools; **#LancsBox**'s GraphColl
does the network version in module 19.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
def kwic(texts, term, width=45, limit=12):
    """Keyword in context: every occurrence with the words either side."""
    pattern = re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)
    rows = []
    for doc_id, text in enumerate(texts):
        s = str(text)
        for m in pattern.finditer(s):
            left = s[max(0, m.start() - width):m.start()].replace("\n", " ")
            right = s[m.end():m.end() + width].replace("\n", " ")
            rows.append({"doc": doc_id, "left": left.rjust(width),
                         "match": m.group(0), "right": right})
            if len(rows) >= limit:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

In [ ]:
TERM = "native"             # ← change this

lines = kwic(df_colonial["text"], TERM, width=45, limit=15)
print(f"{len(lines)} of many occurrences of '{TERM}' in the colonial corpus")
print()
for _, r in lines.iterrows():
    print(f"{r['left']} [{r['match']}] {r['right']}")

## Read down the middle

Do not read the lines as sentences. Read the column immediately left of the
keyword, then the column immediately right. Patterns appear that no individual
sentence shows you.

For *native* in this corpus you will typically see a possessive to the left (*the
native*, *his native*) and a noun to the right (*native chief*, *native
question*) — grammatical evidence that the word is functioning as a category
applied to people rather than a description.

In [ ]:
# The columns, extracted. This is the move that turns reading into evidence.
lines = kwic(df_colonial["text"], TERM, width=40, limit=400)
left_words = Counter(l.split()[-1].lower() for l in lines["left"] if l.split())
right_words = Counter(r.split()[0].lower() for r in lines["right"] if r.split())

print(f"{'immediately BEFORE':<26}{'immediately AFTER'}")
print("-" * 52)
for (lw, ln), (rw, rn) in zip(left_words.most_common(12), right_words.most_common(12)):
    print(f"{lw:<16}{ln:>6}    {rw:<16}{rn:>6}")

### 🔧 Your turn 1

Change `TERM` to a word from your own research — or try `savage`, `civilised`,
`chief`, `mission`.

Read twenty lines. Write one sentence about how the word is used that you could
not have got from its frequency.

## The same tool on contemporary text

KWIC is not only for archives. It is how you check whether a keyness result
means what you think it means.

In [ ]:
NEWS_TERM = "government"    # ← change this

eng = df[df["language"] == "English"]
lines = kwic(eng["text"], NEWS_TERM, width=42, limit=12)
if lines.empty:
    print(f"'{NEWS_TERM}' does not occur in this subset — pick another word.")
    print("(That is worth knowing before you build an analysis on it.)")
for _, r in lines.iterrows():
    print(f"{r['left']} [{r['match']}] {r['right']}")
print()
print(f"{len(kwic(eng['text'], NEWS_TERM, limit=10_000))} occurrences found in total")

In [ ]:
# The check that keyness cannot do for you: is the word being used
# straightforwardly, or negated, opposed, or inside a fixed phrase?
import re
term_lines = kwic(eng["text"], NEWS_TERM, width=60, limit=300)
opposed = sum(1 for l in term_lines.get("left", [])
              if re.search(r"\b(no|not|never|denied|against|anti|oppos\w*|"
                           r"criticis\w*|fight\w*)\b\s*\S*\s*$", l.lower()))
print(f"{opposed} of {len(term_lines)} sampled occurrences have a negation or")
print("opposition word immediately before them.")
print()
print("This is crude — but it is the difference between 'the corpus mentions")
print(f"{NEWS_TERM} often' and 'the corpus is often about OPPOSING it'.")

### 🔧 Your turn 2

Take the highest-keyness word from notebook 04a and run KWIC on it.

Does the context support the interpretation you gave it? This step catches more
errors than any statistical check, and it takes two minutes.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** For *native* the giveaway is the right-hand column: it is almost
always followed by a noun — *native chief*, *native labour*, *native question*,
*native races*. The word is doing classificatory work, not descriptive work. You
can say that from the concordance and you cannot say it from a frequency count,
and it is exactly the kind of observation that turns a count into an argument.

**Your turn 2.** Frequently the answer is "partly". A word that looked like a
theme turns out to appear mostly inside a fixed phrase, an organisation's name,
or a stock formula. That is not a wasted result — "the term appears
overwhelmingly within institutional naming rather than in substantive discussion"
is itself a finding, and one you can only make by reading the lines.

</details>

## Where this stops

You can read the context of any word at scale. Next: **05b — collocation**, which
turns those left and right columns into a systematic measure.